# QML-SleepNet — Stage 05B + Stage 06 — FINAL GUIDE-CORRECTED CONSOLIDATED v1

This is the **single consolidated final Stage06 run**. No further model-development experiments are permitted after this notebook.

## Frozen configuration

### Guide-explicit
- Task A: binary Apnea vs Normal.
- Stage04 8-D QML representation.
- Stage05 causal features + causal-aware attention + intervention/counterfactual regularisation.
- CNN-BiLSTM temporal encoder:
  Conv1D 64 filters with kernels 3/5/7 → BN/ReLU/MaxPool → BiLSTM 128 ×2 → self-attention → global average + max.
- Fusion to ~256-D, then FC 256→128→64→binary output.
- Composite loss weights: 0.60 classification + 0.25 causal + 0.15 upstream/frozen QML regularisation term.
- Focal loss for class imbalance.
- AdamW guide parameters, 10-epoch warm-up, cosine annealing to 1e-6.
- Dropout within 0.3–0.5; fixed here at 0.4.
- Label smoothing 0.1.
- 5-fold CV on the 35 learning records.
- Stage02 per-record z-score plus guide-explicit outlier clipping at ±4 SD.
- Official x01–x35 labels remain sealed.

### Minimal under-specified implementation choices
- Parallel Conv1D branches for kernels 3/5/7.
- MaxPool kernel/stride = 10.
- One self-attention head.
- Focal gamma = 2.
- Fixed 50 epochs, batch size 96.
- Causal-aware gate is applied to the causal16 branch, preserving QML8 unchanged.
- Fold-local focal alpha is derived deterministically from training-fold class counts only.

## Evidence behind the final choices
- Moving the causal-aware gate from QML8 to causal16 improved both tested pilot folds.
- ±4 SD clipping is explicitly in the guide and changes ~1.74% of learning ECG samples; its fold-3 metric effect was small but positive.
- Fold-local focal alpha changed fold-3 accuracy by only +0.03 percentage points but materially improved:
  balanced accuracy (+1.43 pp), F1 (+2.56 pp), AUROC (+0.90 pp), and sensitivity (~+5.4 pp).
  Because headline accuracy was preserved while the other major metrics improved, **alpha is promoted**.

## Workload minimisation
The exact final fold-3 pilot already exists. If its saved NPZ/PT artifacts are present and pass validation,
this notebook reuses fold 3 and trains only folds **0, 1, 2, and 4**, then performs one final all-learning fit.

No threshold search, dropout sweep, IQP tournament, QT substitution, ensemble, temporal smoothing,
scheduler experiment, or architecture search is included.


In [ ]:

# Cell 1 — environment and exact artifact paths

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import gc, json, math, random, time, warnings, shutil
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED=42
FS=100
EPOCH_SECONDS=60
EPOCH_SAMPLES=FS*EPOCH_SECONDS
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ROOT=Path('/content/drive/MyDrive/QML_SleepNet')
S2=ROOT/'data/processed/stage02_preprocessed'

S4=ROOT/'outputs/GUIDE_EXACT_METRICMAX/04_quantum_module_v3_audited'
S4_FOLDS=S4/'stage06_fold_quantum_features'
S4_FINAL=S4/'final_fit/quantum_features_for_final_stage06.npz'

S5=ROOT/'outputs/GUIDE_EXACT_METRICMAX/05_causal_core_v1_2'
S5_FOLDS=S5/'folds'
S5_FINAL=S5/'final_fit/causal16_for_final_stage06.npz'
S5_FINAL_RANK=S5/'final_fit/final_ace_ranking.csv'

OUT=ROOT/'outputs/GUIDE_EXACT_METRICMAX/05B_06_FINAL_GUIDE_CORRECTED_CONSOLIDATED_v1'
FOLD_OUT=OUT/'folds'
FINAL_OUT=OUT/'final_fit'
for p in (OUT,FOLD_OUT,FINAL_OUT): p.mkdir(parents=True,exist_ok=True)

PILOT_F3_ROOT=ROOT/'outputs/GUIDE_EXACT_METRICMAX/05B_06_final_focal_alpha_fold3_pilot_v1'
PILOT_F3_NPZ=PILOT_F3_ROOT/'fold3_causal_gate_clip4_focal_alpha_pilot.npz'
PILOT_F3_PT=PILOT_F3_ROOT/'fold3_causal_gate_clip4_focal_alpha_pilot.pt'

for p in (S2,S4_FOLDS,S4_FINAL,S5_FOLDS,S5_FINAL,S5_FINAL_RANK):
    if not p.exists(): raise FileNotFoundError(p)

print('Device:',DEVICE)
if DEVICE.type!='cuda':
    raise RuntimeError('Use a GPU runtime for Stage06 CNN-BiLSTM training.')


In [ ]:

# Cell 2 — load frozen Stage04 winner + Stage05 final artifacts; keep x labels sealed

qfinal=np.load(S4_FINAL,allow_pickle=False)
cfinal=np.load(S5_FINAL,allow_pickle=True)  # Stage05 UID arrays were saved as dtype=object

UID_LEARN=np.asarray(qfinal['learn_uids']).astype(str)
UID_TEST=np.asarray(qfinal['test_uids']).astype(str)
Y_LEARN=np.asarray(qfinal['y_learn'],dtype=np.int8)

# Stage04 joint-VQC winner: Angle-Rx 8-D quantum measurements.
Q_LEARN=np.asarray(qfinal['vqc_angle8_learn'],dtype=np.float32)
Q_TEST=np.asarray(qfinal['vqc_angle8_test'],dtype=np.float32)

_c_uid_learn=np.asarray(cfinal['learn_uids'])
_c_uid_test=np.asarray(cfinal['test_uids'])
if _c_uid_learn.dtype==object and not all(isinstance(v,str) for v in _c_uid_learn.tolist()):
    raise RuntimeError('Stage05 learn_uids contains non-string objects')
if _c_uid_test.dtype==object and not all(isinstance(v,str) for v in _c_uid_test.tolist()):
    raise RuntimeError('Stage05 test_uids contains non-string objects')

C_UID_LEARN=_c_uid_learn.astype(str)
C_UID_TEST=_c_uid_test.astype(str)
C_LEARN=np.asarray(cfinal['causal16_learn'],dtype=np.float32)
C_TEST=np.asarray(cfinal['causal16_test'],dtype=np.float32)
C_NAMES=np.asarray(cfinal['feature_names']).astype(str)

assert np.array_equal(UID_LEARN,C_UID_LEARN)
assert np.array_equal(UID_TEST,C_UID_TEST)
assert np.array_equal(Y_LEARN,np.asarray(cfinal['y_learn'],dtype=np.int8))
assert Q_LEARN.shape==(len(Y_LEARN),8)
assert Q_TEST.shape==(len(UID_TEST),8)
assert C_LEARN.shape==(len(Y_LEARN),16)
assert C_TEST.shape==(len(UID_TEST),16)
assert 'y_test' not in qfinal.files
assert 'y_test' not in cfinal.files
assert np.isfinite(Q_LEARN).all() and np.isfinite(C_LEARN).all()

print('Learning rows:',len(Y_LEARN))
print('Official x rows:',len(UID_TEST))
print('QML8:',Q_LEARN.shape,'Causal16:',C_LEARN.shape)
print('Official x labels used: NO')


In [ ]:

# Cell 3 — exact five fold packages; essential leakage checks only

FOLDS={}

for fold in range(5):
    qp=S4_FOLDS/f'fold{fold}_quantum_inputs.npz'
    cp=S5_FOLDS/f'fold{fold}_causal16.npz'
    rp=S5_FOLDS/f'fold{fold}_ace_ranking.csv'
    for p in (qp,cp,rp):
        if not p.is_file(): raise FileNotFoundError(p)

    q=np.load(qp,allow_pickle=False)
    c=np.load(cp,allow_pickle=False)
    rank=pd.read_csv(rp)

    tr=np.asarray(q['train_rows'],dtype=np.int64)
    va=np.asarray(q['val_rows'],dtype=np.int64)
    assert np.array_equal(tr,np.asarray(c['train_rows'],dtype=np.int64))
    assert np.array_equal(va,np.asarray(c['val_rows'],dtype=np.int64))
    assert len(set(tr)&set(va))==0
    assert np.array_equal(Y_LEARN[tr],np.asarray(q['y_train'],dtype=np.int8))
    assert np.array_equal(Y_LEARN[va],np.asarray(q['y_val'],dtype=np.int8))
    assert np.array_equal(Y_LEARN[tr],np.asarray(c['y_train'],dtype=np.int8))
    assert np.array_equal(Y_LEARN[va],np.asarray(c['y_val'],dtype=np.int8))

    tr_rec={UID_LEARN[i].rsplit(':',1)[0] for i in tr}
    va_rec={UID_LEARN[i].rsplit(':',1)[0] for i in va}
    if tr_rec & va_rec:
        raise RuntimeError(f'fold {fold}: record leakage detected')
    if (('c05' in tr_rec and 'c06' in va_rec) or ('c06' in tr_rec and 'c05' in va_rec)):
        raise RuntimeError(f'fold {fold}: c05/c06 duplicate-source leakage')

    names=np.asarray(c['feature_names']).astype(str)
    meta=rank.set_index('feature').loc[names]

    FOLDS[fold]={
        'tr':tr,'va':va,
        'ytr':Y_LEARN[tr],'yva':Y_LEARN[va],
        'qtr':np.asarray(q['vqc_angle8_train'],dtype=np.float32),
        'qva':np.asarray(q['vqc_angle8_val'],dtype=np.float32),
        'ctr':np.asarray(c['causal16_train'],dtype=np.float32),
        'cva':np.asarray(c['causal16_val'],dtype=np.float32),
        'names':names,
        'q25':meta['q25_standardized'].to_numpy(np.float32),
        'q75':meta['q75_standardized'].to_numpy(np.float32),
        'ace':meta['ace'].to_numpy(np.float32),
    }

    assert FOLDS[fold]['qtr'].shape[1]==8
    assert FOLDS[fold]['ctr'].shape[1]==16
    assert np.isfinite(FOLDS[fold]['qtr']).all()
    assert np.isfinite(FOLDS[fold]['ctr']).all()
    assert np.isfinite(FOLDS[fold]['ace']).all()

# each learning row must be validation exactly once
counts=np.zeros(len(Y_LEARN),dtype=np.int8)
for d in FOLDS.values(): counts[d['va']]+=1
assert np.all(counts==1)
print('5-fold leakage-safe alignment: PASS')


In [ ]:

# Cell 4 — build the locked 60-second Stage02 ECG bank

def load_record_ecg(rec):
    p=S2/f'{rec}_preprocessed.npz'
    if not p.is_file(): raise FileNotFoundError(p)
    d=np.load(p,allow_pickle=False)
    ecg=np.asarray(d['ecg_filtered'],dtype=np.float32)
    fs=int(np.asarray(d['fs']).item())
    ne=int(np.asarray(d['n_epochs']).item())
    if fs!=FS or len(ecg)!=ne*EPOCH_SAMPLES:
        raise RuntimeError(f'{rec}: Stage02 ECG geometry mismatch')
    mu=float(ecg.mean()); sd=float(ecg.std())
    if not np.isfinite(sd) or sd<1e-8: raise RuntimeError(f'{rec}: invalid ECG std')
    z=((ecg-mu)/sd)
    z=np.clip(z,-4.0,4.0)
    return z.astype(np.float32).reshape(ne,EPOCH_SAMPLES)

def build_ecg_bank(uids):
    X=np.empty((len(uids),1,EPOCH_SAMPLES),dtype=np.float32)
    by_rec={}
    for row,uid in enumerate(uids):
        rec,ep=uid.rsplit(':',1)
        by_rec.setdefault(rec,[]).append((row,int(ep)))
    for rec,items in sorted(by_rec.items()):
        z=load_record_ecg(rec)
        for row,ep in items:
            if ep<0 or ep>=len(z): raise RuntimeError(f'{rec}:{ep} out of range')
            X[row,0]=z[ep]
        del z; gc.collect()
    assert np.isfinite(X).all()
    return X

X_ECG_LEARN=build_ecg_bank(UID_LEARN)
print('Learning ECG:',X_ECG_LEARN.shape)


In [ ]:

# Cell 5 — dataset, reproducibility, metrics

class HybridDataset(Dataset):
    def __init__(self,ecg,qml,causal,y=None):
        self.ecg=ecg; self.qml=qml; self.causal=causal
        self.y=None if y is None else np.asarray(y,dtype=np.int64)
    def __len__(self): return len(self.ecg)
    def __getitem__(self,i):
        x=(torch.from_numpy(self.ecg[i]),torch.from_numpy(self.qml[i]),torch.from_numpy(self.causal[i]))
        if self.y is None: return x
        return (*x,torch.tensor(self.y[i],dtype=torch.long))

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

def metrics(y,p):
    y=np.asarray(y,dtype=np.int8); p=np.asarray(p,dtype=np.float64)
    pred=(p>=0.5).astype(np.int8)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {
        'accuracy':float(accuracy_score(y,pred)),
        'balanced_accuracy':float(balanced_accuracy_score(y,pred)),
        'precision':float(precision_score(y,pred,zero_division=0)),
        'recall_sensitivity':float(recall_score(y,pred,zero_division=0)),
        'specificity':float(tn/max(tn+fp,1)),
        'f1':float(f1_score(y,pred,zero_division=0)),
        'auroc':float(roc_auc_score(y,p)),
        'auprc':float(average_precision_score(y,p)),
        'mcc':float(matthews_corrcoef(y,pred)),
    }


In [ ]:

# Cell 6 — exact Stage06 CNN-BiLSTM blocks + minimal Stage05 causal-aware gate

POOL=10
DROPOUT=0.4

class TemporalEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.c3=nn.Conv1d(1,64,3,padding=1)
        self.c5=nn.Conv1d(1,64,5,padding=2)
        self.c7=nn.Conv1d(1,64,7,padding=3)
        self.bn=nn.BatchNorm1d(192)
        self.pool=nn.MaxPool1d(POOL,POOL)
        self.bilstm=nn.LSTM(192,128,num_layers=2,batch_first=True,bidirectional=True,dropout=DROPOUT)
        self.attn=nn.MultiheadAttention(256,1,dropout=DROPOUT,batch_first=True)
    def forward(self,x):
        h=torch.cat([self.c3(x),self.c5(x),self.c7(x)],dim=1)
        h=self.pool(F.relu(self.bn(h))).transpose(1,2)
        h,_=self.bilstm(h)
        h,_=self.attn(h,h,h,need_weights=False)
        return torch.cat([h.mean(dim=1),h.max(dim=1).values],dim=1)  # 512-D

class GuideHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal=TemporalEncoder()

        # GUIDE-REQUIRED causal-aware gate; target is under-specified by the guide.
        # Minimal last-shot choice: gate the causal branch and preserve QML8 unchanged.
        self.causal_gate=nn.Linear(16,16)

        # QML8 + temporal512 + gated causal16 = 536 -> guide 256-D fusion vector.
        self.fusion256=nn.Linear(536,256)
        self.fc128=nn.Linear(256,128)
        self.fc64=nn.Linear(128,64)
        self.out=nn.Linear(64,2)  # Task A binary
        self.drop=nn.Dropout(DROPOUT)

    def fuse(self,temporal512,qml8,causal16):
        gate=torch.sigmoid(self.causal_gate(causal16))
        causal_gated=causal16*gate
        z=torch.cat([qml8,temporal512,causal_gated],dim=1)
        z=F.relu(self.fusion256(z))
        h=self.drop(F.relu(self.fc128(z)))
        h=self.drop(F.relu(self.fc64(h)))
        return self.out(h)

    def forward(self,ecg,qml8,causal16):
        t=self.temporal(ecg)
        return self.fuse(t,qml8,causal16),t

# fail fast on geometry
_m=GuideHybrid().to(DEVICE)
with torch.no_grad():
    _log,_t=_m(torch.zeros(2,1,EPOCH_SAMPLES,device=DEVICE),torch.zeros(2,8,device=DEVICE),torch.zeros(2,16,device=DEVICE))
assert _log.shape==(2,2) and _t.shape==(2,512)
del _m,_log,_t
if torch.cuda.is_available(): torch.cuda.empty_cache()
print('Model geometry: PASS')



# Loss interpretation

The guide explicitly gives:

`L = 0.60 L_CE + 0.25 L_causal + 0.15 L_QML_regularise`.

- `L_CE`: focal-modulated, label-smoothed classification loss.
- `L_causal`: counterfactual/intervention loss using the Stage-05 Q25/Q75 interventions and ACE estimates.
- `L_QML_regularise`: the VQC is already trained and frozen in Stage04. The guide does not give a downstream
  equation for regularising a frozen 8-D feature vector. To avoid inventing a new penalty, this notebook does **not**
  add a proxy loss. The term is kept as `0.15 × 0` downstream, while the actual Stage04 VQC regularisation/clipping remains upstream.

This is the only honest minimal treatment of that under-specified term without re-running the PennyLane VQC inside
Stage06 or inventing a new objective.


In [ ]:

# Cell 7 — focal CE + Stage05 intervention/counterfactual causal loss

LAMBDA_CE=0.60
LAMBDA_CAUSAL=0.25
LAMBDA_QML=0.15  # upstream/frozen Stage04 VQC term; no invented downstream gradient
LABEL_SMOOTHING=0.1
FOCAL_GAMMA=2.0

def training_class_alpha(y):
    counts=np.bincount(np.asarray(y,dtype=np.int64),minlength=2).astype(np.float64)
    if np.any(counts==0):
        raise RuntimeError(f'Degenerate training labels: {counts}')
    w=len(y)/(2.0*counts)
    w=w/w.mean()
    return torch.tensor(w,dtype=torch.float32,device=DEVICE)

def focal_ce(logits,target,alpha):
    logp=F.log_softmax(logits,dim=1)
    p=logp.exp()
    k=logits.shape[1]
    with torch.no_grad():
        t=torch.full_like(logits,LABEL_SMOOTHING/k)
        t.scatter_(1,target[:,None],1-LABEL_SMOOTHING+LABEL_SMOOTHING/k)
    ce=-(t*logp).sum(dim=1)
    pt=(t*p).sum(dim=1).clamp(1e-7,1-1e-7)
    alpha_t=alpha[target]
    return (alpha_t*(1-pt).pow(FOCAL_GAMMA)*ce).mean()

def causal_loss(model,temporal512,qml8,causal16,q25,q75,ace):
    # Counterfactual augmentation: all 16 Stage05-selected causal features at Q25/Q75.
    # Temporal/QML values are held fixed under do(X_j).
    B,D=causal16.shape
    lo=causal16.unsqueeze(0).repeat(D,1,1)
    hi=causal16.unsqueeze(0).repeat(D,1,1)
    j=torch.arange(D,device=causal16.device)
    lo[j,:,j]=q25[:,None]
    hi[j,:,j]=q75[:,None]
    lo=lo.reshape(D*B,D); hi=hi.reshape(D*B,D)

    t=temporal512.detach().unsqueeze(0).expand(D,-1,-1).reshape(D*B,-1)
    q=qml8.detach().unsqueeze(0).expand(D,-1,-1).reshape(D*B,-1)

    plo=torch.softmax(model.fuse(t,q,lo),dim=1)[:,1].reshape(D,B).mean(dim=1)
    phi=torch.softmax(model.fuse(t,q,hi),dim=1)[:,1].reshape(D,B).mean(dim=1)
    return F.mse_loss(phi-plo,ace)


In [ ]:

# Cell 8 — training and inference

EPOCHS=50
BATCH=96

def train_fold(Xtr,Qtr,Ctr,ytr,Xv,Qv,Cv,yv,q25_np,q75_np,ace_np,seed):
    set_seed(seed)
    model=GuideHybrid().to(DEVICE)

    q25=torch.tensor(q25_np,dtype=torch.float32,device=DEVICE)
    q75=torch.tensor(q75_np,dtype=torch.float32,device=DEVICE)
    ace=torch.tensor(ace_np,dtype=torch.float32,device=DEVICE)
    alpha=training_class_alpha(ytr)
    print('  fold-local focal alpha:',alpha.detach().cpu().numpy().tolist())

    dl=DataLoader(HybridDataset(Xtr,Qtr,Ctr,ytr),batch_size=BATCH,shuffle=True,num_workers=2,pin_memory=True)

    opt=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-4,betas=(0.9,0.999),amsgrad=True,eps=1e-8)

    def lr_lambda(ep):
        if ep<10: return (ep+1)/10
        progress=(ep-10)/max(EPOCHS-11,1)
        r=1e-6/1e-4
        return r+(1-r)*0.5*(1+math.cos(math.pi*progress))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,lr_lambda=lr_lambda)

    for ep in range(EPOCHS):
        model.train(); total=ce_sum=ca_sum=qml_sum=0.0; n=0
        for ecg,qml,causal,y in dl:
            ecg=ecg.to(DEVICE,non_blocking=True); qml=qml.to(DEVICE,non_blocking=True)
            causal=causal.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)

            logits,t=model(ecg,qml,causal)
            Lce=focal_ce(logits,y,alpha)
            Lca=causal_loss(model,t,qml,causal,q25,q75,ace)

            # The Stage04 VQC is frozen here, so there is no trainable downstream
            # VQC parameter on which to apply a new regulariser. Keep the guide's
            # lambda_3 term explicitly present without inventing a proxy objective.
            Lqml=torch.zeros((),dtype=Lce.dtype,device=Lce.device)
            loss=LAMBDA_CE*Lce + LAMBDA_CAUSAL*Lca + LAMBDA_QML*Lqml

            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

            bs=len(y); n+=bs
            total+=float(loss.detach())*bs; ce_sum+=float(Lce.detach())*bs; ca_sum+=float(Lca.detach())*bs; qml_sum+=float(Lqml.detach())*bs

        sched.step()
        if (ep+1)%10==0:
            print(f'  epoch {ep+1:02d}/{EPOCHS} loss={total/n:.5f} ce={ce_sum/n:.5f} causal={ca_sum/n:.5f} qml={qml_sum/n:.5f} lr={opt.param_groups[0]["lr"]:.2e}')

    @torch.no_grad()
    def infer(X,Q,C):
        model.eval(); probs=[]
        dl2=DataLoader(HybridDataset(X,Q,C),batch_size=192,shuffle=False,num_workers=2,pin_memory=True)
        for ecg,qml,causal in dl2:
            ecg=ecg.to(DEVICE,non_blocking=True); qml=qml.to(DEVICE,non_blocking=True); causal=causal.to(DEVICE,non_blocking=True)
            logits,_=model(ecg,qml,causal)
            probs.append(torch.softmax(logits,dim=1)[:,1].cpu().numpy())
        return np.concatenate(probs).astype(np.float64)

    pv=infer(Xv,Qv,Cv) if Xv is not None else None
    return model,(metrics(yv,pv) if pv is not None else None),pv,infer


In [ ]:
# Cell 9 — guide-required 5-fold CV; exact final configuration

rows=[]
OOF=np.full(len(Y_LEARN),np.nan,dtype=np.float64)

# Reuse the exact fold-3 pilot if available and valid.
reuse_f3=False
if PILOT_F3_NPZ.is_file() and PILOT_F3_PT.is_file():
    z=np.load(PILOT_F3_NPZ,allow_pickle=False)
    va=np.asarray(z['val_rows'],dtype=np.int64)
    pv=np.asarray(z['val_probability'],dtype=np.float64)
    met=json.loads(str(z['metrics_json'].item()))

    if not np.array_equal(va,FOLDS[3]['va']):
        raise RuntimeError('Saved final fold-3 pilot validation rows do not match the frozen fold definition')
    if len(pv)!=len(va) or not np.isfinite(pv).all():
        raise RuntimeError('Saved final fold-3 pilot probabilities are invalid')

    ck=torch.load(PILOT_F3_PT,map_location='cpu')
    if int(ck.get('fold',-1))!=3:
        raise RuntimeError('Saved fold-3 pilot checkpoint metadata mismatch')
    if ck.get('gate')!='causal16_to_gate16_on_causal_branch':
        raise RuntimeError('Saved fold-3 pilot gate metadata mismatch')

    cp=FOLD_OUT/'angle_rx_fold3.npz'
    mp=FOLD_OUT/'angle_rx_fold3.pt'
    np.savez_compressed(
        cp,
        val_rows=va,
        val_probability=pv,
        metrics_json=np.asarray(json.dumps(met)),
        seconds=np.asarray(float(z['seconds'].item()) if 'seconds' in z.files else np.nan),
        reused_from=np.asarray(str(PILOT_F3_NPZ)),
    )
    shutil.copy2(PILOT_F3_PT,mp)
    reuse_f3=True
    print('Validated and imported exact final fold-3 pilot:',met)
else:
    print('Exact fold-3 pilot artifacts not found; fold 3 will be trained normally.')

for fold in range(5):
    d=FOLDS[fold]
    cp=FOLD_OUT/f'angle_rx_fold{fold}.npz'
    mp=FOLD_OUT/f'angle_rx_fold{fold}.pt'

    # Safe resume, including imported exact fold 3.
    if cp.is_file() and mp.is_file():
        z=np.load(cp,allow_pickle=False)
        va=np.asarray(z['val_rows'],dtype=np.int64)
        pv=np.asarray(z['val_probability'],dtype=np.float64)
        met=json.loads(str(z['metrics_json'].item()))

        if not np.array_equal(va,d['va']):
            raise RuntimeError(f'fold {fold}: saved validation rows mismatch')
        if len(pv)!=len(va) or not np.isfinite(pv).all():
            raise RuntimeError(f'fold {fold}: saved probabilities invalid')

        OOF[va]=pv
        rows.append({'fold':fold,**met})
        print('RESUME fold',fold,met)
        continue

    tr=d['tr']; va=d['va']
    print(f'\nFINAL GUIDE-CORRECTED HYBRID angle_rx fold={fold} train={len(tr)} val={len(va)}')
    t0=time.time()

    model,met,pv,_=train_fold(
        X_ECG_LEARN[tr],d['qtr'],d['ctr'],d['ytr'],
        X_ECG_LEARN[va],d['qva'],d['cva'],d['yva'],
        d['q25'],d['q75'],d['ace'],SEED+fold
    )

    sec=time.time()-t0
    OOF[va]=pv
    rows.append({'fold':fold,**met})

    np.savez_compressed(
        cp,
        val_rows=va,
        val_probability=pv,
        metrics_json=np.asarray(json.dumps(met)),
        seconds=np.asarray(sec),
    )
    torch.save({
        'state_dict':model.state_dict(),
        'fold':fold,
        'encoding':'angle_rx',
        'gate':'causal16_to_gate16_on_causal_branch',
        'clip_sd':4.0,
        'focal_alpha':'fold_local_inverse_frequency',
        'task':'binary Apnea vs Normal',
        'official_x_labels_used':False,
    },mp)

    print('  metrics:',met)
    print('  seconds:',round(sec,1))

    del model,pv
    gc.collect()
    torch.cuda.empty_cache()

assert np.isfinite(OOF).all()

fold_df=pd.DataFrame(rows).sort_values('fold')
if fold_df.fold.tolist()!=[0,1,2,3,4]:
    raise RuntimeError('5-fold result set incomplete')
fold_df.to_csv(OUT/'five_fold_metrics.csv',index=False)

oof_metrics=metrics(Y_LEARN,OOF)
summary={
    'mean_accuracy':float(fold_df.accuracy.mean()),
    'std_accuracy':float(fold_df.accuracy.std(ddof=1)),
    'mean_balanced_accuracy':float(fold_df.balanced_accuracy.mean()),
    'std_balanced_accuracy':float(fold_df.balanced_accuracy.std(ddof=1)),
    'mean_f1':float(fold_df.f1.mean()),
    'std_f1':float(fold_df.f1.std(ddof=1)),
    'mean_auroc':float(fold_df.auroc.mean()),
    'std_auroc':float(fold_df.auroc.std(ddof=1)),
    'mean_auprc':float(fold_df.auprc.mean()),
    'mean_mcc':float(fold_df.mcc.mean()),
    'oof':oof_metrics,
    'fold3_reused_exact_final_pilot':bool(reuse_f3),
}

print('\nFINAL GUIDE-CORRECTED 5-FOLD SUMMARY')
print(json.dumps(summary,indent=2))

np.savez_compressed(
    OUT/'oof_angle_rx.npz',
    uids=UID_LEARN.astype('U128'),
    y=Y_LEARN,
    probability=OOF,
)
(OUT/'five_fold_summary.json').write_text(json.dumps(summary,indent=2))


In [ ]:

# Cell 10 — final all-learning causal intervention metadata

rank=pd.read_csv(S5_FINAL_RANK).set_index('feature').loc[C_NAMES]
FINAL_Q25=rank['q25_standardized'].to_numpy(np.float32)
FINAL_Q75=rank['q75_standardized'].to_numpy(np.float32)
FINAL_ACE=rank['ace'].to_numpy(np.float32)
assert len(FINAL_Q25)==len(FINAL_Q75)==len(FINAL_ACE)==16
print('Final intervention metadata aligned: PASS')


In [ ]:

# Cell 11 — final all-learning fit; transform official x records without using x labels

X_ECG_TEST=build_ecg_bank(UID_TEST)

FINAL_MODEL,_,_,FINAL_INFER=train_fold(
    X_ECG_LEARN,Q_LEARN,C_LEARN,Y_LEARN,
    None,None,None,None,
    FINAL_Q25,FINAL_Q75,FINAL_ACE,SEED+900
)

P_LEARN=FINAL_INFER(X_ECG_LEARN,Q_LEARN,C_LEARN)
P_TEST=FINAL_INFER(X_ECG_TEST,Q_TEST,C_TEST)

model_path=FINAL_OUT/'qml_sleepnet_final_guide_corrected.pt'
torch.save({
    'state_dict':FINAL_MODEL.state_dict(),
    'encoding':'angle_rx',
    'gate':'causal16_to_gate16_on_causal_branch',
    'clip_sd':4.0,
    'focal_alpha':'all_learning_inverse_frequency',
    'task':'binary Apnea vs Normal',
    'official_x_labels_used':False,
},model_path)

np.savez_compressed(
    FINAL_OUT/'final_predictions.npz',
    learn_uids=UID_LEARN.astype('U128'),
    test_uids=UID_TEST.astype('U128'),
    y_learn=Y_LEARN,
    learn_probability=P_LEARN,
    test_probability=P_TEST,
)

print('Final model:',model_path)
print('Learn probabilities:',P_LEARN.shape)
print('Official x probabilities:',P_TEST.shape)
print('Official x labels used: NO')


In [ ]:
# Cell 12 — final frozen manifest

manifest={
    'pipeline':'QML-SleepNet Stage05B+Stage06 FINAL GUIDE-CORRECTED CONSOLIDATED v1',
    'task':'Task A binary Apnea vs Normal',
    'qml':'frozen Stage04 Angle-Rx VQC8 winner',
    'causal':'frozen Stage05 causal16 + Q25/Q75/ACE interventions',
    'temporal_input':'Stage02 filtered ECG, 60-second minute, per-record z-score, clip to ±4 SD',
    'temporal':'Conv1D64(k3,k5,k7) -> BN/ReLU/MaxPool -> BiLSTM128x2 -> self-attention -> global avg+max',
    'fusion':'causal16 -> gate16 -> gated causal16; QML8 preserved unchanged; QML8 + temporal512 + gated causal16 -> Linear 536->256 -> FC128 -> FC64 -> 2 logits',
    'classification_loss':'label-smoothed focal loss, gamma=2, deterministic fold-local inverse-frequency alpha',
    'composite_loss':'0.60 classification + 0.25 causal intervention + 0.15 frozen/upstream QML term (zero downstream gradient)',
    'training':'AdamW 1e-4, wd 1e-4, betas .9/.999, AMSGrad, eps 1e-8, warmup10 + cosine to 1e-6, dropout .4, epochs50, batch96',
    'validation':'guide-required 5-fold CV on learning records; duplicate-safe; official x labels sealed',
    'fold3_reuse':'exact final pilot reused only if artifact validation passes',
    'not_used':['threshold tuning','probability smoothing','ensemble','IQP tournament','QT64 substitution','dropout sweep','gamma sweep','scheduler experiment','architecture search'],
    'official_x_labels_used':False,
}
(OUT/'FINAL_GUIDE_CORRECTED_MANIFEST.json').write_text(json.dumps(manifest,indent=2))
print(json.dumps(manifest,indent=2))
